In [57]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [58]:
df = pd.read_csv('POS_CASH_balance.csv')

In [59]:
df.shape

(10001358, 8)

In [60]:
df.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10001358 entries, 0 to 10001357
Data columns (total 8 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   SK_ID_PREV             int64  
 1   SK_ID_CURR             int64  
 2   MONTHS_BALANCE         int64  
 3   CNT_INSTALMENT         float64
 4   CNT_INSTALMENT_FUTURE  float64
 5   NAME_CONTRACT_STATUS   object 
 6   SK_DPD                 int64  
 7   SK_DPD_DEF             int64  
dtypes: float64(2), int64(5), object(1)
memory usage: 610.4+ MB


In [62]:
df.duplicated().sum()

np.int64(0)

Check SK_ID_PERV

In [63]:
df.SK_ID_PREV.nunique()

936325

In [64]:
df.SK_ID_PREV.is_unique

False

Check SK_ID_CURR

In [65]:
df.SK_ID_CURR.nunique()

337252

In [66]:
df.SK_ID_CURR.is_unique

False

Check Categorical Values

In [67]:
df.NAME_CONTRACT_STATUS.value_counts()

,count
NAME_CONTRACT_STATUS,
Active,9151119
Completed,744883
Signed,87260
Demand,7065
Returned to the store,5461
Approved,4917
Amortized debt,636
Canceled,15
XNA,2


Missing Percentage

In [68]:
missing_pct = (
    df.isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missing_pct

,0
CNT_INSTALMENT_FUTURE,0.260835
CNT_INSTALMENT,0.260675
SK_ID_CURR,0.000000
SK_ID_PREV,0.000000
MONTHS_BALANCE,0.000000
NAME_CONTRACT_STATUS,0.000000
SK_DPD,0.000000
SK_DPD_DEF,0.000000


Create a monthly delinquency flag

In [69]:
df['HAS_DPD'] = (
    df['SK_DPD'] > 0
).astype(int)

In [70]:
# ============================================================
# 1. LOAN-LEVEL AGGREGATION
# ============================================================

loan_level = (
    df.groupby(['SK_ID_CURR', 'SK_ID_PREV'])
      .agg(
          LOAN_MONTHS=('MONTHS_BALANCE', 'count'),

          LOAN_AVG_INST=('CNT_INSTALMENT', 'mean'),
          LOAN_AVG_FUTURE_INST=('CNT_INSTALMENT_FUTURE', 'mean'),

          LOAN_MAX_DPD=('SK_DPD', 'max'),
          LOAN_MAX_DPD_DEF=('SK_DPD_DEF', 'max'),

          LOAN_DPD_MONTHS=('HAS_DPD', 'sum')
      )
      .reset_index()
)


# DPD frequency for each loan
loan_level['LOAN_DPD_RATE'] = (
    loan_level['LOAN_DPD_MONTHS'] /
    loan_level['LOAN_MONTHS'].replace(0, np.nan)
)

In [71]:
loan_level.head()

,SK_ID_CURR,SK_ID_PREV,LOAN_MONTHS,LOAN_AVG_INST,LOAN_AVG_FUTURE_INST,LOAN_MAX_DPD,LOAN_MAX_DPD_DEF,LOAN_DPD_MONTHS,LOAN_DPD_RATE
0,100001,1369693,5,4.000,2.000,0,0,0,0.00
1,100001,1851984,4,4.000,0.750,7,7,1,0.25
2,100002,1038818,19,24.000,15.000,0,0,0,0.00
3,100003,1810518,8,11.375,7.875,0,0,0,0.00
4,100003,2396755,12,12.000,6.500,0,0,0,0.00


In [72]:
# ============================================================
# 2. CUSTOMER-LEVEL AGGREGATION
# ============================================================

customer_features = (
    loan_level
    .groupby('SK_ID_CURR')
    .agg(
        POS_CASH_LOANS=('SK_ID_PREV', 'nunique'),

        POS_CASH_AVG_LOAN_MONTHS=('LOAN_MONTHS', 'mean'),
        POS_CASH_MAX_LOAN_MONTHS=('LOAN_MONTHS', 'max'),

        POS_CASH_AVG_INST=('LOAN_AVG_INST', 'mean'),
        POS_CASH_AVG_FUTURE_INST=('LOAN_AVG_FUTURE_INST', 'mean'),

        POS_CASH_MAX_DPD=('LOAN_MAX_DPD', 'max'),
        POS_CASH_AVG_DPD=('LOAN_MAX_DPD', 'mean'),

        POS_CASH_TOTAL_DPD_MONTHS=('LOAN_DPD_MONTHS', 'sum'),

        POS_CASH_AVG_DPD_RATE=('LOAN_DPD_RATE', 'mean'),
        POS_CASH_MAX_DPD_RATE=('LOAN_DPD_RATE', 'max'),

        POS_CASH_MAX_DPD_DEF=('LOAN_MAX_DPD_DEF', 'max')
    )
    .reset_index()
)

Add contract-status information

In [73]:
# Start with counts:
status_counts = pd.crosstab(
    df['SK_ID_CURR'],
    df['NAME_CONTRACT_STATUS']
)

In [74]:
# Rename them:
status_counts.columns = [
    f'POS_CASH_STATUS_{col.upper().replace(" ", "_")}_COUNT'
    for col in status_counts.columns
]

In [75]:
# Remove extremely tiny categories:
status_counts = status_counts.drop(
    columns=[
        'POS_CASH_STATUS_CANCELED_COUNT',
        'POS_CASH_STATUS_XNA_COUNT',
        'POS_CASH_STATUS_AMORTIZED_COUNT',
        'POS_CASH_STATUS_APPROVED_COUNT'
    ],
    errors='ignore'
)

In [76]:
# Then join:
final_pos_cash_features = (
    customer_features
    .join(status_counts, how='left')
)

In [84]:
final_pos_cash_features = final_pos_cash_features.fillna(0)

# final_pos_cash_features = (
#     final_pos_cash_features
#     .reset_index()
# )

In [85]:
final_pos_cash_features.head(10)

,SK_ID_CURR,POS_CASH_LOANS,POS_CASH_AVG_LOAN_MONTHS,POS_CASH_MAX_LOAN_MONTHS,POS_CASH_AVG_INST,POS_CASH_AVG_FUTURE_INST,POS_CASH_MAX_DPD,POS_CASH_AVG_DPD,POS_CASH_TOTAL_DPD_MONTHS,POS_CASH_AVG_DPD_RATE,POS_CASH_MAX_DPD_RATE,POS_CASH_MAX_DPD_DEF,POS_CASH_STATUS_ACTIVE_COUNT,POS_CASH_STATUS_AMORTIZED_DEBT_COUNT,POS_CASH_STATUS_COMPLETED_COUNT,POS_CASH_STATUS_DEMAND_COUNT,POS_CASH_STATUS_RETURNED_TO_THE_STORE_COUNT,POS_CASH_STATUS_SIGNED_COUNT
0,100001,2,4.500000,5,4.000000,1.375000,7,3.5,1,0.125000,0.250000,7,0.0,0.0,0.0,0.0,0.0,0.0
1,100002,1,19.000000,19,24.000000,15.000000,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100003,3,9.333333,12,9.791667,5.666667,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100004,1,4.000000,4,3.750000,2.250000,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
4,100005,1,11.000000,11,11.700000,7.200000,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
5,100006,3,7.000000,10,12.888889,10.214286,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
6,100007,5,13.200000,18,15.066667,8.966667,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
7,100008,4,20.750000,55,13.388889,8.050821,1294,323.5,43,0.195455,0.781818,0,0.0,0.0,0.0,0.0,0.0,0.0
8,100009,8,8.000000,13,7.625000,4.149038,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0
9,100010,1,11.000000,11,10.000000,5.000000,0,0.0,0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0.0,0.0


Validate

In [86]:
print("Shape:", final_pos_cash_features.shape)

print(
    "Unique customers:",
    final_pos_cash_features['SK_ID_CURR'].nunique()
)

print(
    "Duplicate customers:",
    final_pos_cash_features['SK_ID_CURR'].duplicated().sum()
)

print("\nMissing values:")
display(
    final_pos_cash_features
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(5)
)

Shape: (337252, 18)
Unique customers: 337252
Duplicate customers: 0

Missing values:


,0
SK_ID_CURR,0
POS_CASH_LOANS,0
POS_CASH_AVG_LOAN_MONTHS,0
POS_CASH_MAX_LOAN_MONTHS,0
POS_CASH_AVG_INST,0


**SAVE IT**

In [87]:
final_pos_cash_features.to_csv(
    'pos_cash_aggreagate.csv',
    index=False
)